# MPT

In [1]:
# Import Libraries
import warnings
warnings.filterwarnings('ignore')

#import data manipulation libraries
import pandas as pd
import numpy as np
from numpy.linalg import multi_dot
import yfinance as yf
from sympy import Matrix
import sqlalchemy
import matplotlib.pyplot as plt

#Set numpy random seed
np.random.seed(42)

#Import cufflinks for visualization
import cufflinks as cf
cf.set_config_file(offline=True, dimensions=((1000,600)))

#Import plotly express
import plotly.express as px
import plotly.graph_objects as go
px.defaults.width, px.defaults.height = 1000,600

#set precision
pd.set_option('display.precision', 4)

In [2]:
### Creating a database for storage
engine = sqlalchemy.create_engine('sqlite:///India')

In [3]:
# Read data from wikipedia - refer Lab 1 for further details
nifty50 = pd.read_html('https://en.wikipedia.org/wiki/NIFTY_50')[2].Symbol.to_list()
# Fetch data from yahoo using list comprehension
data = [yf.download(symbol+'.NS', start="2019-01-01", end="2023-12-31", progress=False).reset_index() for symbol in nifty50]
# save it to database
for frame, symbol in zip(data, nifty50):
    frame.to_sql(symbol, engine, if_exists='replace', index=False)

In [4]:
 # Specify assets / stocks
# Indian stocks : bank, consumer goods, diversified, it, consumer durables 
assets = sorted(['ICICIBANK', 'ITC', 'RELIANCE', 'TCS', 'ASIANPAINT']) 
print(assets)
# Number of assets
numofasset = len(assets)
# Number of portfolio for optimization
numofportfolio = 5000

['ASIANPAINT', 'ICICIBANK', 'ITC', 'RELIANCE', 'TCS']


In [5]:
 # Query close price from database
df = pd.DataFrame()
for asset in assets:
    df1 = pd.read_sql_query(f'SELECT Date, Close FROM {asset}', engine, index_col='Date')
    df1.columns = [asset]
    df = pd.concat([df, df1], axis=1)
# View output
df

,ASIANPAINT,ICICIBANK,ITC,RELIANCE,TCS
Date,,,,,
2019-01-01 00:00:00.000000,1371.5500,363.75,282.70,1024.9669,1902.8000
2019-01-02 00:00:00.000000,1383.3000,364.60,280.60,1011.6177,1923.3000
2019-01-03 00:00:00.000000,1388.3000,363.25,278.85,999.1371,1899.9500
2019-01-04 00:00:00.000000,1385.8500,365.20,280.95,1004.5316,1876.8500
2019-01-07 00:00:00.000000,1396.0000,367.70,281.65,1010.1091,1897.9000
...,...,...,...,...,...
2023-12-22 00:00:00.000000,3341.3000,994.30,455.20,2565.0500,3824.0000
2023-12-26 00:00:00.000000,3383.3501,995.10,456.45,2578.0500,3795.5500
2023-12-27 00:00:00.000000,3404.4500,1002.25,457.10,2586.8501,3811.2000


### Visualize Time series

In [6]:
# Plot price history
df['2019':].normalize().iplot(kind='line')

In [7]:
 # Dataframe of returns and volatility
returns = df.pct_change().dropna()
annual_returns = round(returns.mean()*260*100,2)
annual_stdev = round(returns.std()*np.sqrt(260)*100,2)
# Subsume into dataframe
df2 = pd.DataFrame({
    'Ann Ret': annual_returns,
    'Ann Vol': annual_stdev
})
# Get the output
df2

,Ann Ret,Ann Vol
ASIANPAINT,22.73,26.75
ICICIBANK,26.85,33.41
ITC,13.96,26.78
RELIANCE,24.24,30.89
TCS,17.72,25.22


In [8]:
 # Plot annualized return and volatility
df2.iplot(
    kind='bar',
    shared_xaxes=True,
    orientation='h'
)

### Portfolio Composition

In [9]:
df2.reset_index().iplot(
    kind='pie',
    labels='index',
    values='Ann Ret',
    textinfo='percent+label',
    hole=0.6
)

# PORTFOLIO STATISTICS

In [10]:
def portfolio_simulation(returns): # initialize the lists
    rets=[]; vols=[]; wts=[] # simulate 5000 portfolio
    for i in range(numofportfolio): 
        # generate random weights
        weights = np.random.random(numofasset)
        # set weights such that sum of weights equals 1
        weights /= np.sum(weights)
        # portfolio stats
        rets.append(weights.T@np.array(returns.mean()*260)) #@ is for dot product
        vols.append(np.sqrt(multi_dot([weights.T, returns.cov()*260, weights])))
        wts.append(weights)
        
    # create a datafrme for analysis
    data = {'port_rets': rets, 'port_vols': vols}
    for counter, symbol in enumerate(returns.columns.tolist()):
        data[symbol+' weight'] = [w[counter] for w in wts]
    portdf = pd.DataFrame(data)
    portdf['sharpe_ratio'] = portdf['port_rets'] / portdf['port_vols']
    return round(portdf,4)

### Maximize Sharpe Ratio

In [11]:
# Create a dataframe for analysis
temp = portfolio_simulation(returns)
temp.head()

,port_rets,port_vols,ASIANPAINT weight,ICICIBANK weight,ITC weight,RELIANCE weight,TCS weight,sharpe_ratio
0,0.2189,0.2125,0.1332,0.3381,0.2603,0.2129,0.0555,1.0299
1,0.1855,0.1918,0.0653,0.0243,0.3625,0.2516,0.2963,0.9668
2,0.2097,0.2269,0.0093,0.4375,0.3755,0.0958,0.0820,0.9244
3,0.2034,0.1961,0.1057,0.1753,0.3024,0.2489,0.1678,1.0372
4,0.2074,0.1887,0.3279,0.0748,0.1566,0.1963,0.2444,1.0992


In [12]:
 # Get the max sharpe portfolio stats
temp.iloc[temp.sharpe_ratio.idxmax()]

port_rets            0.2184
port_vols            0.1953
ASIANPAINT weight    0.3292
ICICIBANK weight     0.1820
ITC weight           0.0831
RELIANCE weight      0.1720
TCS weight           0.2336
sharpe_ratio         1.1182
Name: 553, dtype: float64

In [13]:
# Verify the above result
temp.describe().T

,count,mean,std,min,25%,50%,75%,max
port_rets,5000.0,0.2110,0.0132,0.1614,0.2024,0.2110,0.2196,0.2537
port_vols,5000.0,0.2016,0.0117,0.1822,0.1933,0.1996,0.2078,0.2788
ASIANPAINT weight,5000.0,0.2012,0.1140,0.0000,0.1124,0.2001,0.2785,0.7079
ICICIBANK weight,5000.0,0.1973,0.1130,0.0000,0.1082,0.1985,0.2755,0.7488
ITC weight,5000.0,0.1996,0.1128,0.0000,0.1112,0.2009,0.2768,0.7678
RELIANCE weight,5000.0,0.2030,0.1119,0.0001,0.1155,0.2029,0.2805,0.7058
TCS weight,5000.0,0.1988,0.1117,0.0001,0.1103,0.1982,0.2752,0.6781
sharpe_ratio,5000.0,1.0477,0.0459,0.7027,1.0235,1.0570,1.0814,1.1182


### Visualize Simulated Portfolio

In [14]:
# Plot simulated portfolio
fig = px.scatter(
    temp, x='port_vols', y='port_rets', color='sharpe_ratio',
    labels={'port_vols': 'Expected Volatility', 'port_rets': 'Expected Return','sharpe_ratio': 'Sharpe Ratio'}, title="Monte Carlo Simulated Portfolio"
     ).update_traces(mode='markers', marker=dict(symbol='circle'))
# Plot max sharpe
fig.add_scatter(
    mode='markers',
    x=[temp.iloc[temp.sharpe_ratio.idxmax()]['port_vols']],
    y=[temp.iloc[temp.sharpe_ratio.idxmax()]['port_rets']],
    marker=dict(color='RoyalBlue', size=20, symbol='star'),
    name = 'Max Sharpe'
).update(layout_showlegend=False)
# Show spikes
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)
fig.show()

# EFFICIENT FRONTIER

## Constrained Optimization

In [15]:
# Import optimization module from scipy 
# sco.minimize?
import scipy.optimize as sco
sco.minimize

<function scipy.optimize._minimize.minimize(fun, x0, args=(), method=None, jac=None, hess=None, hessp=None, bounds=None, constraints=(), tol=None, callback=None, options=None)>

## Portfolio Statistics

In [16]:
def portfolio_stats(weights):
    weights = np.array(weights)
    port_rets = weights.T @ np.array(returns.mean() * 260)
    port_vols = np.sqrt(multi_dot([weights.T, returns.cov() * 260, weights]))
    return np.array([port_rets, port_vols, port_rets/port_vols])
# Minimize the volatility
def min_volatility(weights):
    return portfolio_stats(weights)[1]
# Minimize the variance
def min_variance(weights):
    return portfolio_stats(weights)[1]**2
# Maximizing sharpe ratio
def max_sharpe_ratio(weights):
    return -portfolio_stats(weights)[2]

## Efficient Frontier Portfolio

In [17]:
 # Specify constraints, bounds and initial weights
cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bnds = tuple((0,1) for x in range(numofasset))
initial_wts = numofasset*[1./numofasset]

In [18]:
# Optimizing for maximum sharpe ratio
opt_sharpe = sco.minimize(max_sharpe_ratio, initial_wts, method='SLSQP', bounds=bnds, constraints=cons)
# Optimizing for minimum variance
opt_var = sco.minimize(min_variance, initial_wts, method='SLSQP', bounds=bnds,constraints=cons)

In [19]:
opt_sharpe

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -1.1186600513818665
       x: [ 3.251e-01  1.968e-01  8.503e-02  1.770e-01  2.162e-01]
     nit: 5
     jac: [-1.373e-04  5.700e-04 -2.605e-04 -5.494e-04  2.398e-04]
    nfev: 30
    njev: 5

In [20]:
opt_var

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: 0.03313696252928448
       x: [ 2.553e-01  4.434e-02  2.944e-01  8.664e-02  3.193e-01]
     nit: 7
     jac: [ 6.631e-02  6.630e-02  6.639e-02  6.622e-02  6.615e-02]
    nfev: 42
    njev: 7

In [21]:
# Efficient Frontier
targetrets = np.linspace(0.15,0.24,100)
tvols = []
for tr in targetrets:
    ef_cons = ({'type': 'eq', 'fun': lambda x: portfolio_stats(x)[0] - tr},
               {'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    opt_ef = sco.minimize(min_volatility, initial_wts, method='SLSQP', bounds=bnds, constraints=ef_cons)
    tvols.append(opt_ef['fun'])
targetvols = np.array(tvols)

In [22]:
# Create EF Dataframe for plotting
efport = pd.DataFrame({
    'targetrets' : np.around(100*targetrets,2),
    'targetvols': np.around(100*targetvols,2),
    'targetsharpe': np.around(targetrets/targetvols,2)
})
efport.head()

,targetrets,targetvols,targetsharpe
0,15.00,21.95,0.68
1,15.09,21.66,0.70
2,15.18,21.39,0.71
3,15.27,21.14,0.72
4,15.36,20.93,0.73


In [23]:
# Plot efficient frontier portfolio
fig = px.scatter(
    efport, x='targetvols', y='targetrets',  color='targetsharpe',
    labels={'targetrets': 'Expected Return', 'targetvols': 'Expected␣Volatility','targetsharpe': 'Sharpe Ratio'}, title="Efficient Frontier Portfolio" ).update_traces(mode='markers', marker=dict(symbol='cross'))
# Plot maximum sharpe portfolio
fig.add_scatter(
    mode='markers',
    x=[100*portfolio_stats(opt_sharpe['x'])[1]],
    y=[100*portfolio_stats(opt_sharpe['x'])[0]],
    marker=dict(color='red', size=20, symbol='circle'),
    name = 'Max Sharpe'
).update(layout_showlegend=False)
# Plot minimum variance portfolio
fig.add_scatter(
    mode='markers',
    x=[100*portfolio_stats(opt_var['x'])[1]],
    y=[100*portfolio_stats(opt_var['x'])[0]],
    marker=dict(color='green', size=20, symbol='star'),
    name = 'Min Variance').update(layout_showlegend=False)
# Show spikes
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)
fig.show()
#fig.write_image_html("images/ef.png")
#fig.write_image('Efficient-frontier.png')

# MODULE 2 LECTURE 2 EXERCISE

## STARTS

## Question 1

### Define the shared inputs

In [24]:
global mu, sigmaP, unit_1

mu = np.array([0.08,0.10,0.10,0.14])
sigmaP = np.diag(np.array([0.12, 0.12, 0.15, 0.20]))
unit_1 = np.ones((4,))
#one = np.array([1 , 1, 1, 1])

Recall Define parameters used in Lagrange

$\Sigma = S^{T} R S $ - where R is rho ($\rho)$\
$ A = 1^{T}\Sigma^{-1}1 $ 
$ B = \mu^{T}\Sigma^{-1}1 $
$ C = \mu^{T}\Sigma^{-1}\mu $
$ \lambda = \frac{A\times m - B}{A \times C - B^{2} $
$ \gamma = \frac{C - B\times m}{A \times C - B^{2} $
$ W^{*} = \Sigma^{-1} (\lambda\times\mu + \gamma * \vec{1})$



### Case 1

In [25]:
rho1 = np.array([[1,0.2,0.5,0.3],
                 [0.2,1,0.7,0.4],
                 [0.5,0.7,1,0.9],
                 [0.3,0.4,0.9,1]])

cov1 = sigmaP @ rho1 @ sigmaP #Covariance matrix
#cov = multi_dot([sigmaP, rho, sigmaP]) #alternatively multidot
InVcov1 = np.linalg.inv(cov1)
A1 = unit_1.T @ InVcov1 @ unit_1
B1 = mu.T @ InVcov1 @ unit_1
C1 = mu.T @ InVcov1 @ mu
print(A1, B1, C1)

456.5217391304311 63.81642512077241 9.38969404186788


In [26]:
#Function to determine the optimal weights provided with the mean, A, B,C and Sigma paramaters
def Opt_Weights(meantgt, A, B, C, Sigma):
    m = meantgt
    lmbda = (A * m - B)/(A * C - pow(B,2))
    gmma = (C - B*m)/(A * C - pow(B,2))
    Wtarget1 = Sigma@(lmbda*mu + gmma*unit_1)
    return Wtarget1

In [27]:
m = 0.1
Case1_weights = Opt_Weights(m, A1, B1, C1, InVcov1)
Case1_weights

array([ 0.76228686,  0.84419926, -0.98762956,  0.38114343])

### Case 2

In [28]:
rho2 = np.diag(np.array([1, 1, 1, 1]))

cov2 = sigmaP @ rho2 @ sigmaP #Covariance matrix
InVcov2 = np.linalg.inv(cov2)
A2 = unit_1.T @ InVcov2 @ unit_1
B2 = mu.T @ InVcov2 @ unit_1
C2 = mu.T @ InVcov2 @ mu

In [29]:
Case2_weights = Opt_Weights(m, A2, B2, C2,InVcov2)
Case2_weights

array([0.29827662, 0.33694211, 0.21564295, 0.14913831])

### Case 3

In [30]:
rho3 = np.ones((4,4),)
cov3 = sigmaP @ rho3 @ sigmaP #Covariance matrix
InVcov3 = np.linalg.inv(cov3)
A3 = unit_1.T @ InVcov3 @ unit_1
B3 = mu.T @ InVcov3 @ unit_1
C3 = mu.T @ InVcov3 @ mu

LinAlgError: Singular matrix

In [ ]:
# Express the variance
VarP1 = lambda x : (A1 * pow(x,2) - 2*B1*x + C1)/(A1 * C1 - pow(B1,2))
y1 = np.linspace(0, 0.3, 1000)  #Creating a range of returns
x1 = np.sqrt(VarP1(y1))

## Question 2b

In [ ]:
M = Matrix([[9,3,0],[3,16,5],[0,5,25]])
M

In [ ]:
M.inv()

In [ ]:
rhob = Matrix([[1,0.2,0.5,0.3],[0.2,1,0.7,0.4],[0.5,0.7,1,0.9],[0.3,0.4,0.9,1]])
cov1 = sigmaP @ rhob @ sigmaP
cov1

## Question 4

PARAMETERS
Expected return Asset A, $\mu_{a} = 10\% $\
Expected return Asset B, $\mu_{b} = 20\% $
Standard Deviation of return for Asset A, $\sigma_{a} = 0.2 $
Standard Deviation of return for Asset B, $\sigma_{b} = 0.3 $
Correlation of Assets A and B, $\rho_{ab} = 0.5 $
Risk free asset annual return, $R_{f} = 5\% $

##### Calculate $W_{a}$

$W_{a} = \frac{E_{ra}\sigma_{b}^{2} - E_{rb}\rho_{ab}\times\sigma_{a}\times\sigma_{b}}
{E_{rb}\sigma_{a}^{2} + E_{ra}\sigma_{b}^{2} -  \rho_{AB}\times\sigma_{a}\times\sigma_{b}\times[E_{ra}+E_{ra}]} $


Where $ {E_{ra}} = \mu_{a} - R_{f}$
and    $ {E_{rb}} = \mu_{b} - R_{f}$

In [ ]:
mua = 0.10
mub = 0.20
SDa = 0.2
SDb = 0.3
CorrelAB = 0.5
rf = 0.05
CovAB = CorrelAB * SDa * SDb

Wa = ((mua-rf)*SDb**2 - (mub-rf)*CovAB)/((mua-rf)*SDb**2 + (mub-rf)*SDa**2 - (CovAB*((mua-rf)+(mub-rf))))
Wb = 1- Wa
RtnPort = (Wa*mua) + (Wb*mub)
SDevPort = np.sqrt((Wa**2 * SDa**2) + (Wb**2 * SDb**2) - (2*Wa*Wb*CovAB))

print(RtnPort)
print(SDevPort)

In [ ]:
Wstr = np.linspace(-2.5,1,200)
mupi = lambda x : (0.20 - x * 0.10)
sdpi = lambda x : np.sqrt((0.07*(pow(x,2))) - 0.12*x + 0.09)

x_axis = sdpi(Wstr)
y_axis = mupi(Wstr)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_axis, y=y_axis, mode='lines', name="Que4"))
fig.update_layout(title='Efficient Frontier',
                  xaxis_title='risk, $\sigma$',
                  yaxis_title='return, $\mu$',
                  autosize=True,
                  xaxis_range=[0, max(x_axis)],
                  yaxis_range=[-0.10,max(y_axis)],
                  width=720,
                  height=600)
fig.show()

Capital Market Line (CML

$\mu_{p} = r_{f} + \theta_{t} \times \sigma{p} $\
$\theta_{t} = \frac{\mu_{t} - r_{f}}{\sigma_{t}} $

we can either:
$ \underset{W}{max}\theta(W) = \frac{\mu_{\Pi} - r_{f}}{\sigma_{\Pi}} $

or 
we minimize the funcion:
$ \underset{w}{min} \frac{1}{2}w^{T}\Sigma w $

subject to:
$ r_{f} + (\mu - r_{f}\times \vec{1})^{T} w = m $

Derived $W^{*} = \frac{\Sigma^{-1}(\mu - r_{f})\times \vec{1}}{B - A\times r_{f}}$

In [ ]:
rhoAB = np.array([[1,0.5],[0.5,1]])
SDab = np.diag(np.array([SDa, SDb]))
CovP = SDab @ rhoAB @ SDab
InVcovP = np.linalg.inv(CovP)
unit_1b = np.array([1,1])
Mu = np.array([mua, mub])
A4 = unit_1b.T @ InVcovP @ unit_1b
B4 = Mu.T @ InVcovP @ unit_1b

In [ ]:
Wt = InVcovP@((Mu - (rf*unit_1b)))/(B4 - (A4*rf))
Wt

Tangency Portfolio is basically asset B:
$\mu_{t} = \mu^{T} \times w_{t} $
$ \sigma_{t} = \sqrt{w_{t}^{T}\Sigma w_{t}} $

In [ ]:
Mut = Mu @ Wt
Sigmat = np.sqrt(Wt.T@CovP@Wt)
Sharpe_t = (Mut - rf)/Sigmat


The equation of the CML is:
$\mu{p} = 0.05 + 0.5 \times \sigma{p} $

In [ ]:
x_axis2 = np.linspace(0,1,200)
MuP = lambda x: (0.05 + 0.5 * x)
y_axis2 = MuP(x_axis2)

### Replot Efficient frontier plus CML

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_axis, y=y_axis, mode='lines', name="Efficient frontier"))
fig.add_trace(go.Scatter(x=x_axis2, y=y_axis2, mode='lines', name="CML"))
fig.update_layout(title='Question 4 M2L2 Exercise',
                  xaxis_title='risk, $\sigma$',
                  yaxis_title='return, $\mu$',
                  autosize=True,
                  xaxis_range=[0, max(x_axis)],
                  yaxis_range=[-0.10,max(y_axis2)],
                  width=720,
                  height=600)
fig.show()

## Question 3

In [ ]:
Mat2a = Matrix(([9,3,0], [3,16,5], [0,5,25]))
Mat2a

In [ ]:
Mat2aDeterminant = Mat2a.det()
MatCofactors = Mat2a.cofactor_matrix()
Mat2aInv = Mat2a.inv()
Mat2aDeterminant

In [ ]:
MatCofactors

In [ ]:
Mat2aInv

## Question 3

$ W_{t} = \frac{\Sigma^{-1}(\mu - r \vec{1}}{B - A\times r} $

In [ ]:
rstr = 0.05
uppr = InVcov1 @ (mu - rstr * unit_1)
dwnr = B1 - A1*rstr
W_rmin = uppr/dwnr
W_rmin

In [ ]:
w0 = unit_vector.T @ W_rmin